# Feature Extraction Baseline

This notebook extracts image features (SigLIP/DINOv2) for downstream classification experiments.

Dependencies on previous stages:
- `../temporal/results/temporal_pred.csv` (temporal predictions)
- `../spatial/results/molmo_pred_analysis.pkl` (spatial predictions)

Run from this directory to keep all relative paths valid.
Outputs are written under `./results/`.

In [29]:
%matplotlib inline

import re
from transformers import AutoImageProcessor, AutoModel, AutoProcessor
from tqdm import tqdm
from PIL import Image
import pandas as pd
import os
import sys
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm

sys.path.append('../..')
from reasoning.utils import get_frame_by_id, get_every_nth_frame, crop_with_bbox, crop_around_point_pixels
from reasoning.visualize import visualize_points_on_image

class DinoV2Extractor():
    def __init__(self, model_name='facebook/dinov2-base'):
        self.model_name = model_name
        self.processor = AutoImageProcessor.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)

    def __call__(self, images):
        inputs = self.processor(images=images, return_tensors="pt")
        outputs = self.model(**inputs)
        return outputs.pooler_output


class Siglip2Extractor():
    def __init__(self, model_name="google/siglip2-base-patch16-224", device='cpu'):
        self.model = AutoModel.from_pretrained(model_name, device_map=device).eval()
        self.processor = AutoProcessor.from_pretrained(model_name)
        self.model_name = model_name

    def __call__(self, images):
        inputs = self.processor(images=images, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            image_embeddings = self.model.get_image_features(**inputs)    
        return image_embeddings


def get_features(extractor, df, col_frame='accident_frame', spatial_oracle=True):
    results = []
    for i, row in tqdm(df.iterrows(), total=len(df)):
        data = row.to_dict()    
        img_gt = get_frame_by_id(row['video_path'], row[col_frame])

        if spatial_oracle:
            crop_fixed = crop_with_bbox(
                img_gt,
                bbox=(row['x1']*row['width'], row['y1']*row['height'], row['x2']*row['width'], row['y2']*row['height']),
                padding_percent=0.3
            )
        else:
            crop_fixed = crop_around_point_pixels(
                img_gt,
                point=(row['x_pred'], row['y_pred']),
                crop_size=(224, 224)
            )
        crop_full =  img_gt
    
        with torch.no_grad():
            data['feats'] = extractor([crop_fixed, crop_full])
        results.append(data)
    return results


DATA_DIR = '../../data'


df = pd.read_csv(f'{DATA_DIR}/labels.csv')
df['video_path'] = DATA_DIR + '/' + df['path']

# Add temporal results
df_pred = pd.read_csv('../temporal/results/temporal_pred.csv')
df['pred_frame_qwen'] = df_pred['pred_frame_qwen']
df['pred_frame_molmo'] = df_pred['pred_frame_molmo']

# Add spatial results
spatial = torch.load(f"../spatial/results/molmo_pred_analysis.pkl", weights_only=False)
x_pred = pd.Series([i['spatial']['x'] for i in spatial])
df['x_pred'] = x_pred.fillna(df['width'] / 2).astype(int)
y_pred = pd.Series([i['spatial']['y'] for i in spatial])
df['y_pred'] = y_pred.fillna(df['height'] / 2).astype(int)

# Load sim dataset
df_sim = pd.read_csv(f"{DATA_DIR}/sim_dataset/labels.csv")
df_sim['video_path'] = DATA_DIR + "/sim_dataset/" + df_sim['rgb_path']

os.makedirs(f'results/', exist_ok=True)

# Feature extraction
- Extract features using Dino2 and Siglip models
- Use either predicted frame, predicted point or ground truth data

## Siglip

In [30]:
siglip_extractor = Siglip2Extractor()

# Molmo preds
results = get_features(siglip_extractor, df, col_frame='pred_frame_molmo', spatial_oracle=False)
torch.save(results, 'results/feats_pred_molmo_siglip.pkl')

# Oracle
results = get_features(siglip_extractor, df, col_frame='accident_frame')
torch.save(results, 'results/feats_oracle_siglip.pkl')

# Sim dataset
results = get_features(siglip_extractor, df_sim, col_frame='accident_frame')
torch.save(results, 'results/feats_sim_siglip.pkl')

100%|██████████| 2027/2027 [09:55<00:00,  3.40it/s]


## Dino v2

In [31]:
dino_extractor = DinoV2Extractor()

# Molmo preds
results = get_features(dino_extractor, df, col_frame='pred_frame_molmo', spatial_oracle=False)
torch.save(results, 'results/feats_pred_molmo_dinov2.pkl')

# Oracle
results = get_features(dino_extractor, df, col_frame='accident_frame')
torch.save(results, 'results/feats_oracle_dinov2.pkl')

# Sim dataset
results = get_features(dino_extractor, df_sim, col_frame='accident_frame')
torch.save(results, 'results/feats_sim_dinov2.pkl')


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
100%|██████████| 2211/2211 [19:11<00:00,  1.92it/s]
